In [105]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# Ignoro i warning 
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Percorso da cui prendere il file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Serve per far paritre il debug
DEBUG = False

# Numero di ripetizioni per allenare
N_REPEAT = 1

STAMPA_POST_BILANCIAMENTO = True

# Target

In [106]:
marker = ["PR", "ER"]

# Bilancio il dataset in fase di training

In [107]:
def undersample_set(X, y, random_state):
    """Bilancia il dataset prendendo n_min campioni per classe"""
    
    # forza nome target
    y = y.copy()
    y.name = "target"

    # Unisco feature e target
    df_tmp = pd.concat([X, y], axis=1)

    # Separazione classi
    df_0 = df_tmp[df_tmp["target"] == 0]
    df_1 = df_tmp[df_tmp["target"] == 1]

    # Dimensione classe minore
    n_min = min(len(df_0), len(df_1))

    if STAMPA_POST_BILANCIAMENTO:
        print(f"    PRIMA del bilanciamento: Classe 0: {len(df_0)}, Classe 1: {len(df_1)}")
    
    # Campionamento casuale
    df_bal = pd.concat([
        df_0.sample(n=n_min, random_state=random_state),
        df_1.sample(n=n_min, random_state=random_state)
    ])

    # Shuffle finale
    df_bal = df_bal.sample(frac=1, random_state=random_state)

    # Split finale
    X_bal = df_bal.drop(columns=["target"])
    y_bal = df_bal["target"]

    # Stampa DOPO il bilanciamento
    if STAMPA_POST_BILANCIAMENTO:
        print(f"    DOPO il bilanciamento:  Classe 0: {(y_bal == 0).sum()}, Classe 1: {(y_bal == 1).sum()}")

    return X_bal, y_bal

# Creo fold bilanciate manualmente

In [108]:
def create_balanced_cv_splits(X, y, n_splits=5, random_state=42):
    """
    Crea split per cross-validation dove ogni test set è bilanciato.
    Il training set contiene il resto dei dati (sbilanciato).
    """
    np.random.seed(random_state)
    
    # Resetta gli indici per evitare problemi
    X_reset = X.reset_index(drop=True)
    y_reset = y.reset_index(drop=True)
    
    # Separa gli indici per classe
    idx_0 = y_reset[y_reset == 0].index.to_numpy()
    idx_1 = y_reset[y_reset == 1].index.to_numpy()
    
    # Shuffle
    np.random.shuffle(idx_0)
    np.random.shuffle(idx_1)

    # Trova la classe con meno campioni
    n_min = min(len(idx_0), len(idx_1))
    n_test_per_class = n_min // n_splits
    
    print(f"  Classe 0: {len(idx_0)} campioni")
    print(f"  Classe 1: {len(idx_1)} campioni")
    print(f"  Test set per fold: {n_test_per_class} × 2 = {n_test_per_class * 2} campioni bilanciati")
    
    splits = []
    for fold in range(n_splits):
        # Indici per il test set (bilanciato)
        start_idx = fold * n_test_per_class
        end_idx = (fold + 1) * n_test_per_class
        
        test_idx_0 = idx_0[start_idx:end_idx]
        test_idx_1 = idx_1[start_idx:end_idx]
        test_idx = np.concatenate([test_idx_0, test_idx_1])
        
        # Tutti gli altri indici vanno nel training
        all_idx = np.arange(len(y_reset))
        train_idx = np.array([i for i in all_idx if i not in test_idx])
        
        splits.append((train_idx, test_idx))
    
    return splits, X_reset, y_reset

# Controllo il numero delle classi

In [109]:
# Carico il csv e stampo la shape
df_duke = pd.read_csv(FILE_PATH / "duke_lesions.csv")
print("DUKE shape:", df_duke.shape)

# Controllo il numero delle classi
for m in marker:
    vc = df_duke[m].value_counts(dropna=False)
    print(f"\nDistribuzione {m} – DUKE")
    print("-" * 30)
    print("Negativi:", vc.get(0, 0))
    print("Positivi:", vc.get(1, 0))
    print("Totale pazienti:", len(df_duke))

# Seleziono le features
FEATURES = [c for c in df_duke.columns if c.startswith("original_")]
print(f"\nNumero features: {len(FEATURES)}")

DUKE shape: (291, 109)

Distribuzione PR – DUKE
------------------------------
Negativi: 157
Positivi: 134
Totale pazienti: 291

Distribuzione ER – DUKE
------------------------------
Negativi: 123
Positivi: 168
Totale pazienti: 291

Numero features: 105


# Stratified Cross-Validation

In [110]:
FEATURES = [c for c in df_duke.columns if c.startswith("original_")]

# Training

In [111]:
results_rows = []

for target in marker:
    print("\n" + "="*80)
    print(f"TARGET: {target}")
    print("="*80)

    X = df_duke[FEATURES]
    y = df_duke[target]

    print("Distribuzione originale:", dict(y.value_counts()))

    # Creo gli split bilanciati per il test set
    cv_splits, X_reset, y_reset = create_balanced_cv_splits(X, y, n_splits=5, random_state=42)

    acc_scores, bal_scores, f1_scores, auc_scores = [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(cv_splits, start=1):

        X_train_raw = X_reset.iloc[train_idx]
        y_train_raw = y_reset.iloc[train_idx]
        X_test = X_reset.iloc[test_idx]
        y_test = y_reset.iloc[test_idx]

        # Verifica bilanciamento
        print("\n" + "="*50)
        print(f"FOLD {fold}")
        print("="*50)
        print("PRIMA DEL BILANCIAMENTO:")
        print(f"  Train set - Classe 0: {(y_train_raw == 0).sum()}, Classe 1: {(y_train_raw == 1).sum()}")
        print(f"  Test set  - Classe 0: {(y_test == 0).sum()}, Classe 1: {(y_test == 1).sum()}")

        all_preds = []
        all_probs = []
        
        print(f"\nADDESTRAMENTO ENSEMBLE ({N_REPEAT} modelli):")
        # Ensemble dei 10 modelli
        for i in range(N_REPEAT):
            print(f"  Modello {i+1}/{N_REPEAT}:")
            
            # Bilancia il training set con seed diverso
            X_train, y_train = undersample_set(
                X_train_raw, 
                y_train_raw, 
                random_state=42 + fold * 100 + i
            )

            model = XGBClassifier(
                random_state=42,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=3,
                reg_alpha=0,
                reg_lambda=1,
                n_estimators=100,      
                max_depth=3,           
                learning_rate=0.1,     
                n_jobs=-1,
                verbosity=0
            )

            model.fit(X_train, y_train, verbose=False)

            all_preds.append(model.predict(X_test))
            all_probs.append(model.predict_proba(X_test)[:, 1])

        all_preds = np.array(all_preds)
        all_probs = np.array(all_probs)

        y_pred = (np.mean(all_preds, axis=0) >= 0.5).astype(int)
        y_prob = np.mean(all_probs, axis=0)

        acc = accuracy_score(y_test, y_pred)
        bal = balanced_accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)

        acc_scores.append(acc)
        bal_scores.append(bal)
        f1_scores.append(f1)
        auc_scores.append(auc)

        print(f"\nRISULTATI FOLD {fold}:")
        print(f"  Accuracy: {acc:.4f} | Balanced Accuracy: {bal:.4f}")
        print(f"  Differenza: {abs(acc - bal):.6f}")
        print(classification_report(y_test, y_pred, zero_division=0))

    # RISULTATI FINALI
    print("\n" + "-"*50)
    print(f"RISULTATI MEDI FINALI - {target}")
    print(f"Accuracy : {np.mean(acc_scores):.3f} ± {np.std(acc_scores):.3f}")
    print(f"Balanced : {np.mean(bal_scores):.3f} ± {np.std(bal_scores):.3f}")
    print(f"F1-score : {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
    print(f"ROC-AUC  : {np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}")
    print("-"*50)

    results_rows.append({
        "Dataset": "DUKE",
        "Target": target,
        "Accuracy": f"{np.mean(acc_scores):.3f} ± {np.std(acc_scores):.3f}",
        "Balanced Accuracy": f"{np.mean(bal_scores):.3f} ± {np.std(bal_scores):.3f}",
        "F1-score": f"{np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}",
        "ROC-AUC": f"{np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}"
    })

results_df = pd.DataFrame(results_rows)
display(results_df)


TARGET: PR
Distribuzione originale: {0: np.int64(157), 1: np.int64(134)}
  Classe 0: 157 campioni
  Classe 1: 134 campioni
  Test set per fold: 26 × 2 = 52 campioni bilanciati

FOLD 1
PRIMA DEL BILANCIAMENTO:
  Train set - Classe 0: 131, Classe 1: 108
  Test set  - Classe 0: 26, Classe 1: 26

ADDESTRAMENTO ENSEMBLE (1 modelli):
  Modello 1/1:
    PRIMA del bilanciamento: Classe 0: 131, Classe 1: 108
    DOPO il bilanciamento:  Classe 0: 108, Classe 1: 108

RISULTATI FOLD 1:
  Accuracy: 0.4423 | Balanced Accuracy: 0.4423
  Differenza: 0.000000
              precision    recall  f1-score   support

           0       0.44      0.46      0.45        26
           1       0.44      0.42      0.43        26

    accuracy                           0.44        52
   macro avg       0.44      0.44      0.44        52
weighted avg       0.44      0.44      0.44        52


FOLD 2
PRIMA DEL BILANCIAMENTO:
  Train set - Classe 0: 131, Classe 1: 108
  Test set  - Classe 0: 26, Classe 1: 26

ADDES

,Dataset,Target,Accuracy,Balanced Accuracy,F1-score,ROC-AUC
0,DUKE,PR,0.496 ± 0.037,0.496 ± 0.037,0.505 ± 0.039,0.504 ± 0.021
1,DUKE,ER,0.496 ± 0.091,0.496 ± 0.091,0.484 ± 0.090,0.530 ± 0.059


# STESSO APPROCCIO MA PER AMBL

# Funzioni

In [112]:
def undersample_set_ambl(X, y, random_state):
    """Bilancia il dataset prendendo n_min campioni per classe"""
    y = y.copy()
    y.name = "target"
    
    df_tmp = pd.concat([X, y], axis=1)
    df_0 = df_tmp[df_tmp["target"] == 0]
    df_1 = df_tmp[df_tmp["target"] == 1]
    
    n_min = min(len(df_0), len(df_1))
    
    if STAMPA_POST_BILANCIAMENTO:
        print(f"    PRIMA del bilanciamento: Classe 0: {len(df_0)}, Classe 1: {len(df_1)}")
    
    df_bal = pd.concat([
        df_0.sample(n=n_min, random_state=random_state),
        df_1.sample(n=n_min, random_state=random_state)
    ])
    
    df_bal = df_bal.sample(frac=1, random_state=random_state)
    
    X_bal = df_bal.drop(columns=["target"])
    y_bal = df_bal["target"]
    
    if STAMPA_POST_BILANCIAMENTO:
        print(f"    DOPO il bilanciamento:  Classe 0: {(y_bal == 0).sum()}, Classe 1: {(y_bal == 1).sum()}")
    
    return X_bal, y_bal


def create_balanced_cv_splits_ambl(X, y, n_splits=5, random_state=42):
    """
    Crea split per cross-validation dove ogni test set è bilanciato.
    """
    np.random.seed(random_state)
    
    X_reset = X.reset_index(drop=True)
    y_reset = y.reset_index(drop=True)
    
    idx_0 = y_reset[y_reset == 0].index.to_numpy()
    idx_1 = y_reset[y_reset == 1].index.to_numpy()
    
    np.random.shuffle(idx_0)
    np.random.shuffle(idx_1)
    
    n_min = min(len(idx_0), len(idx_1))
    n_test_per_class = n_min // n_splits
    
    print(f"  Classe 0: {len(idx_0)} campioni")
    print(f"  Classe 1: {len(idx_1)} campioni")
    print(f"  Test set per fold: {n_test_per_class} × 2 = {n_test_per_class * 2} campioni bilanciati")
    
    splits = []
    for fold in range(n_splits):
        start_idx = fold * n_test_per_class
        end_idx = (fold + 1) * n_test_per_class
        
        test_idx_0 = idx_0[start_idx:end_idx]
        test_idx_1 = idx_1[start_idx:end_idx]
        test_idx = np.concatenate([test_idx_0, test_idx_1])
        
        all_idx = np.arange(len(y_reset))
        train_idx = np.array([i for i in all_idx if i not in test_idx])
        
        splits.append((train_idx, test_idx))
    
    return splits, X_reset, y_reset

# Target

In [113]:
Marker = ["PR [SII]"]

# Controllo il numero di classi

In [114]:
# Carico il csv e stampo la shape
df_ambl = pd.read_csv(FILE_PATH / "ambl_lesions.csv")
print("DUKE shape:", df_ambl.shape)

# Controllo il numero delle classi
for m in Marker:
    vc = df_ambl[m].value_counts(dropna=False)
    print(f"\nDistribuzione {m} – DUKE")
    print("-" * 30)
    print("Negativi:", vc.get(0, 0))
    print("Positivi:", vc.get(1, 0))
    print("Totale pazienti:", len(df_ambl))

# Seleziono le features
FEATURES = [c for c in df_ambl.columns if c.startswith("original_")]
print(f"\nNumero features: {len(FEATURES)}")

DUKE shape: (82, 111)

Distribuzione PR [SII] – DUKE
------------------------------
Negativi: 31
Positivi: 2
Totale pazienti: 82

Numero features: 105


# Binarizzo il tutto

In [115]:
df_ambl_clean = df_ambl.dropna(subset=Marker).copy() 
print(f"\nShape dopo dropna: {df_ambl_clean.shape}")
print(f"Righe rimosse: {len(df_ambl) - len(df_ambl_clean)}")

# Converti in binario (>= 1 diventa 1, < 1 diventa 0)
for m in Marker:
    df_ambl_clean[m] = (pd.to_numeric(df_ambl_clean[m]) >= 1).astype(int)



print("\n" + "="*80)
print("DISTRIBUZIONE CLASSI DOPO PULIZIA")
print("="*80)
for m in Marker:
    vc = df_ambl_clean[m].value_counts()
    print(f"\n{m}:")
    print(f"  Negativi (0): {vc.get(0, 0)}")
    print(f"  Positivi (1): {vc.get(1, 0)}")
    print(f"  Totale: {len(df_ambl_clean)}")
    ratio = max(vc) / min(vc) if min(vc) > 0 else float('inf')
    print(f"  Sbilanciamento: {ratio:.2f}x")


Shape dopo dropna: (67, 111)
Righe rimosse: 15

DISTRIBUZIONE CLASSI DOPO PULIZIA

PR [SII]:
  Negativi (0): 31
  Positivi (1): 36
  Totale: 67
  Sbilanciamento: 1.16x


# Feature

In [116]:
FEATURES = [c for c in df_ambl_clean.columns if c.startswith("original_")]
print(f"\nNumero features: {len(FEATURES)}")



Numero features: 105


# Training

In [117]:
results_rows = []

for target in Marker:
    print("\n" + "="*80)
    print(f"TARGET: {target}")
    print("="*80)

    X = df_ambl_clean[FEATURES]
    y = df_ambl_clean[target]

    print(f"Distribuzione: {dict(y.value_counts())}")

    # Crea gli split bilanciati
    cv_splits, X_reset, y_reset = create_balanced_cv_splits_ambl(
        X, y, n_splits=5, random_state=42
    )

    acc_scores, bal_scores, f1_scores, auc_scores = [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(cv_splits, start=1):

        X_train_raw = X_reset.iloc[train_idx]
        y_train_raw = y_reset.iloc[train_idx]
        X_test = X_reset.iloc[test_idx]
        y_test = y_reset.iloc[test_idx]

        print(f"\n{'='*50}")
        print(f"FOLD {fold}")
        print(f"{'='*50}")
        print(f"Train - Classe 0: {(y_train_raw==0).sum()}, Classe 1: {(y_train_raw==1).sum()}")
        print(f"Test  - Classe 0: {(y_test==0).sum()}, Classe 1: {(y_test==1).sum()}")

        all_preds = []
        all_probs = []

        print(f"\nADDESTRAMENTO ENSEMBLE ({N_REPEAT} MODELLI)")

        for i in range(N_REPEAT):

            X_train, y_train = undersample_set(
                X_train_raw,
                y_train_raw,
                random_state=42 + fold*100 + i
            )


            model = XGBClassifier(
                random_state=42,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=3,
                reg_alpha=0,
                reg_lambda=1,
                n_estimators=100,      
                max_depth=3,           
                learning_rate=0.1,  
                n_jobs=-1,
                verbosity=0
            )

            model.fit(X_train, y_train)

            all_preds.append(model.predict(X_test))
            all_probs.append(model.predict_proba(X_test)[:,1])

        all_preds = np.array(all_preds)
        all_probs = np.array(all_probs)

        # Media ensemble
        y_pred = (np.mean(all_preds, axis=0) >= 0.5).astype(int)
        y_prob = np.mean(all_probs, axis=0)

        acc = accuracy_score(y_test, y_pred)
        bal = balanced_accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)

        acc_scores.append(acc)
        bal_scores.append(bal)
        f1_scores.append(f1)
        auc_scores.append(auc)

        print(f"\nAccuracy: {acc:.4f} | Balanced Accuracy: {bal:.4f}")
        print(f"Differenza: {abs(acc - bal):.6f}")
        print(classification_report(y_test, y_pred, zero_division=0))

    # RISULTATI FINALI
    print("\n" + "-"*50)
    print(f"RISULTATI MEDI FINALI - {target}")
    print(f"Accuracy : {np.mean(acc_scores):.3f} ± {np.std(acc_scores):.3f}")
    print(f"Balanced : {np.mean(bal_scores):.3f} ± {np.std(bal_scores):.3f}")
    print(f"F1-score : {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
    print(f"ROC-AUC  : {np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}")
    print("-"*50)

    results_rows.append({
        "Dataset": "AMBL",
        "Target": target,
        "Accuracy": f"{np.mean(acc_scores):.3f} ± {np.std(acc_scores):.3f}",
        "Balanced Accuracy": f"{np.mean(bal_scores):.3f} ± {np.std(bal_scores):.3f}",
        "F1-score": f"{np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}",
        "ROC-AUC": f"{np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}"
    })

results_df = pd.DataFrame(results_rows)
display(results_df)



TARGET: PR [SII]
Distribuzione: {1: np.int64(36), 0: np.int64(31)}
  Classe 0: 31 campioni
  Classe 1: 36 campioni
  Test set per fold: 6 × 2 = 12 campioni bilanciati

FOLD 1
Train - Classe 0: 25, Classe 1: 30
Test  - Classe 0: 6, Classe 1: 6

ADDESTRAMENTO ENSEMBLE (1 MODELLI)
    PRIMA del bilanciamento: Classe 0: 25, Classe 1: 30
    DOPO il bilanciamento:  Classe 0: 25, Classe 1: 25

Accuracy: 0.5833 | Balanced Accuracy: 0.5833
Differenza: 0.000000
              precision    recall  f1-score   support

           0       0.60      0.50      0.55         6
           1       0.57      0.67      0.62         6

    accuracy                           0.58        12
   macro avg       0.59      0.58      0.58        12
weighted avg       0.59      0.58      0.58        12


FOLD 2
Train - Classe 0: 25, Classe 1: 30
Test  - Classe 0: 6, Classe 1: 6

ADDESTRAMENTO ENSEMBLE (1 MODELLI)
    PRIMA del bilanciamento: Classe 0: 25, Classe 1: 30
    DOPO il bilanciamento:  Classe 0: 25, Class

,Dataset,Target,Accuracy,Balanced Accuracy,F1-score,ROC-AUC
0,AMBL,PR [SII],0.600 ± 0.111,0.600 ± 0.111,0.608 ± 0.127,0.558 ± 0.168
